# 🧠 Template 2 : modèle simple — embeddings + petit réseau

*Préparé par Claude le 19/10/2026 pour le projet `01-AT&T_spam_detector.ipynb`. Aucun code de solution ici : c'est toi qui écris.*

## Les 4 templates du projet

| # | Phase | Notebook | Entrée | Sortie |
|---|---|---|---|---|
| 1 | Préparation des données | `02-Template_1_preparation_donnees` | `src/spam.csv` | `src/train.csv`, `val.csv`, `test.csv` |
| **2** | **Modèle simple : embeddings + petit réseau** (ce notebook) | `03-Template_2_modele_embeddings` | les 3 CSV | `scores/scores_embeddings.json`, `scores/predictions_embeddings.csv` |
| 3 | Transfer learning : DistilBERT | `04-Template_3_transfer_learning` | les 3 CSV | `scores/scores_transfer.json`, `scores/predictions_transfer.csv` |
| 4 | Évaluation et conclusion | `05-Template_4_evaluation_conclusion` | les fichiers de `scores/` | le tableau final du livrable |

Ce notebook répond au premier conseil de l'énoncé : **« Start simple »**. On construit un petit réseau de neurones **de zéro**, en PyTorch, qui n'apprend qu'à partir de nos 3 617 SMS d'entraînement.

## Le principe

```
"FREE entry NOW!"  →  ["free", "entry", "now"]  →  [57, 912, 33, 0, 0, …, 0]  →  Embedding  →  moyenne  →  petit réseau  →  P(spam)
      texte              mots (étape 2)            numéros, 40 cases (3-4)       1 vecteur       1 vecteur     (étape 6)       0 à 1
                                                                                  par mot         par SMS
```

Un **embedding**, c'est une table qui associe à chaque mot du vocabulaire un vecteur de nombres (ici 32). Au départ, ces nombres sont au hasard ; pendant l'entraînement, le réseau les ajuste pour que les mots « de spam » (*free*, *txt*, *claim*...) se retrouvent du même côté.

## Comment s'en servir

Même rythme qu'au template 1 :

1. **🎯 Objectif** : ce qu'on veut obtenir, et pourquoi ;
2. **🧰 Outils** : les fonctions utiles, leur rôle, leurs arguments importants (parfois avec une petite démo dans un autre contexte) ;
3. **✍️ À toi** : une cellule avec le plan en commentaires, que tu remplis ;
4. **✅ Contrôle** : une cellule déjà écrite qui vérifie ton travail. ✅ = c'est bon, `AssertionError` = relis l'étape.

Les noms de variables imposés sont ceux qu'attendent les contrôles. Lance toujours les cellules **dans l'ordre**.

⏱️ L'entraînement complet prend quelques secondes sur un processeur ordinaire : n'hésite pas à expérimenter.

---
# Étape 0 — Les imports
Rien à écrire : lance la cellule.

In [ ]:
import json
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)

DOSSIER_SRC = Path("src")          # train.csv, val.csv, test.csv (sortie du template 1)
DOSSIER_SCORES = Path("scores")    # scores et prédictions : partagés sur GitHub pour le template 4
DOSSIER_MODELES = Path("models")   # modèles entraînés : ignorés par Git (voir .gitignore)
SEED = 42                          # graine du hasard, la même partout
MAX_LEN = 40                       # longueur des séquences, choisie au template 1 (étape 3.c)
MIN_FREQ = 2                       # un mot doit apparaître au moins 2 fois dans train pour entrer au vocabulaire

torch.manual_seed(SEED)
np.random.seed(SEED)
print("prêt — PyTorch", torch.__version__)

---
# Étape 1 — Charger les trois paquets

🎯 Lire les trois CSV préparés au template 1 dans `train_df`, `val_df`, `test_df`.

Ils sont en UTF-8 (le défaut) : plus besoin de `encoding=`.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `pd.read_csv(chemin)` | lit un CSV | — |
| `DOSSIER_SRC / "train.csv"` | construit le chemin du fichier | — |

In [ ]:
# ✍️ À toi
# 1. train_df, val_df, test_df = les fichiers train.csv, val.csv, test.csv du DOSSIER_SRC
# 2. affiche la forme de chacun et train_df.head()

In [ ]:
# ✅ Contrôle
assert (len(train_df), len(val_df), len(test_df)) == (3617, 776, 776), "relance le template 1 : les CSV n'ont pas la bonne taille"
assert list(train_df.columns) == ["text", "label"]
print("✅ étape 1")

---
# Étape 2 — Découper un SMS en mots (tokeniser)

🎯 Écrire une fonction `tokeniser(texte)` qui renvoie la **liste des mots** d'un SMS, en 3 gestes :

1. remplacer `&lt;#&gt;` par le mot `" numtoken "` (la décision du template 1 : un nombre anonymisé devient **un seul mot**) ;
2. tout passer en **minuscules** (*FREE*, *Free* et *free* sont le même mot) ;
3. garder les suites de lettres et de chiffres, sans la ponctuation.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `texte.replace(ancien, nouveau)` | remplace un morceau de texte | — |
| `texte.lower()` | passe en minuscules | — |
| `re.findall(r"\w+", texte)` | liste des suites de « caractères de mot » (lettres, chiffres, `_`) | `r"..."` : chaîne brute, pour que `\w` reste intact |

Démo dans un autre contexte :

In [ ]:
phrase = "Bonjour, ÇA va ? 2 cafés !"
print(re.findall(r"\w+", phrase.lower()))   # la ponctuation disparaît, les accents restent

In [ ]:
# ✍️ À toi
# 1. def tokeniser(texte): les 3 gestes ci-dessus, dans cet ordre (le replace AVANT le lower)
# 2. essaie-la sur le premier SMS de train_df

In [ ]:
# ✅ Contrôle
assert tokeniser("Call NOW &lt;#&gt; free!!") == ["call", "now", "numtoken", "free"], tokeniser("Call NOW &lt;#&gt; free!!")
assert tokeniser("") == []
print("✅ étape 2")

❓ **Pourquoi le `replace` avant le `lower` ?** Ici ça marcherait dans les deux sens, mais prends l'habitude : on remplace un motif **exact** avant de modifier le texte, sinon le motif risque de ne plus être reconnu.

---
# Étape 3 — Construire le vocabulaire

🎯 Un réseau ne lit pas des mots, il lit des **numéros**. On fabrique un dictionnaire `vocab` : mot → numéro.

| Numéro | Mot | Rôle |
|---|---|---|
| 0 | `"<pad>"` | le **remplissage** (*padding*) qui complète les SMS trop courts |
| 1 | `"<unk>"` | tout mot **inconnu** (*unknown*) |
| 2, 3, 4... | les mots vus **au moins `MIN_FREQ` fois** dans train, du plus fréquent au moins fréquent | |

⚠️ **Seulement sur `train_df`.** Val et test jouent le rôle de SMS jamais vus : en production, un mot nouveau sera inconnu du modèle, il doit en être de même ici. Construire le vocabulaire sur tout le jeu serait une petite **fuite de données**.

❓ Pourquoi `MIN_FREQ = 2` ? Un mot vu **une seule fois** (faute de frappe, numéro de téléphone) ne permet pas au modèle d'apprendre quoi que ce soit : il alourdit le vocabulaire pour rien.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `Counter()` | un compteur vide | — |
| `compteur.update(liste)` | ajoute 1 pour chaque élément de la liste | — |
| `compteur.most_common()` | liste des `(mot, nombre)`, du plus fréquent au moins fréquent | — |
| `enumerate(liste, start=2)` | numérote les éléments à partir de 2 | `start=2` |

Démo dans un autre contexte :

In [ ]:
fruits = Counter()
fruits.update(["pomme", "kiwi", "pomme"])
fruits.update(["pomme", "poire"])
print(fruits.most_common())                                 # [('pomme', 3), ('kiwi', 1), ('poire', 1)]
print({f: i for i, (f, n) in enumerate(fruits.most_common(), start=2) if n >= 2})

In [ ]:
# ✍️ À toi
# 1. compteur = un Counter de tous les mots de tous les SMS de train_df (utilise tokeniser)
# 2. vocab = {"<pad>": 0, "<unk>": 1}, puis chaque mot vu au moins MIN_FREQ fois, numéroté à partir de 2,
#    du plus fréquent au moins fréquent
#    ⚠️ attention à la démo : il faut d'abord GARDER les mots fréquents, PUIS les numéroter (sinon il y a des trous)
# 3. affiche le nombre de mots différents dans train, la taille de vocab et les 15 premiers mots

In [ ]:
# ✅ Contrôle
assert vocab["<pad>"] == 0 and vocab["<unk>"] == 1
assert sorted(vocab.values()) == list(range(len(vocab))), "les numéros doivent se suivre sans trou : 0, 1, 2, 3..."
assert list(vocab)[2] == compteur.most_common(1)[0][0], "le numéro 2 doit être le mot le plus fréquent"
assert len(vocab) == 3177, f"3177 entrées attendues (2 spéciales + 3175 mots), obtenu {len(vocab)}"
print("✅ étape 3")

👀 **Observe** : combien de mots différents dans train, et combien en garde-t-on ? Les mots les plus fréquents sont-ils intéressants pour repérer un spam ?

---
# Étape 4 — Transformer chaque SMS en 40 numéros

🎯 Écrire `encoder(texte)` qui renvoie une liste d'**exactement `MAX_LEN` = 40 numéros** :

1. tokeniser le texte ;
2. remplacer chaque mot par son numéro (`1` s'il n'est pas dans `vocab`) ;
3. **couper** à 40 si c'est trop long ;
4. **compléter** avec des `0` à la fin si c'est trop court.

Puis encoder les trois paquets en **tenseurs** PyTorch :
- `X_train`, `X_val`, `X_test` : entiers (`torch.long`), forme `(nombre de SMS, 40)` ;
- `y_train`, `y_val`, `y_test` : les labels en **décimaux** (`torch.float32`), car la fonction d'erreur de l'étape 7 attend des nombres décimaux.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `vocab.get(mot, 1)` | le numéro du mot, ou 1 s'il est absent | — |
| `liste[:MAX_LEN]` | garde les `MAX_LEN` premiers éléments | — |
| `[0] * n` | une liste de `n` zéros | — |
| `torch.tensor(données, dtype=...)` | transforme une liste (de listes) en tenseur | `torch.long`, `torch.float32` |

Démo dans un autre contexte :

In [ ]:
notes = [12, 15, 9]
print(notes[:5] + [0] * (5 - len(notes[:5])))                  # complète jusqu'à 5 cases
print(torch.tensor([[1, 2], [3, 4]], dtype=torch.long).shape)   # 2 lignes, 2 colonnes

In [ ]:
# ✍️ À toi
# 1. def encoder(texte): les 4 gestes ci-dessus → liste de MAX_LEN entiers
# 2. X_train, X_val, X_test = tenseurs torch.long des SMS encodés de chaque paquet
# 3. y_train, y_val, y_test = tenseurs torch.float32 des labels
# 4. affiche les formes, et un SMS de train à côté de son encodage

In [ ]:
# ✅ Contrôle
assert encoder("zzqxw") == [1] + [0] * (MAX_LEN - 1), "un mot inconnu doit donner 1, puis des 0"
assert len(encoder("call " * 100)) == MAX_LEN, "un SMS trop long doit être coupé à MAX_LEN"
assert X_train.shape == (3617, MAX_LEN) and X_train.dtype == torch.long
assert X_test.shape == (776, MAX_LEN)
assert y_train.dtype == torch.float32 and int(y_train.sum()) == 457
taux_unk = (X_val == 1).sum() / (X_val > 0).sum()
print(f"✅ étape 4 — {taux_unk:.1%} des mots de val sont inconnus (<unk>)")

---
# Étape 5 — Préparer les lots (DataLoader)

🎯 Le réseau n'apprend pas sur les 3 617 SMS d'un coup, mais par **lots** (*batches*) de 32 : il corrige ses poids après chaque lot, soit 114 corrections par passage sur train (une **époque**).

- `train_loader` : lots de 32, **mélangés** à chaque époque (sinon le réseau verrait toujours les SMS dans le même ordre) ;
- `val_loader` et `test_loader` : lots de 32, **sans** mélange (on veut garder l'ordre pour comparer aux vrais labels).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `TensorDataset(X, y)` | associe chaque SMS encodé à son label | — |
| `DataLoader(dataset, ...)` | découpe en lots | `batch_size=32`, `shuffle=True` / `False` |
| `next(iter(loader))` | le premier lot, pour regarder | — |

In [ ]:
# ✍️ À toi
# 1. train_loader, val_loader, test_loader (batch_size=32 ; shuffle seulement pour train)
# 2. xb, yb = le premier lot de train_loader ; affiche leurs formes

In [ ]:
# ✅ Contrôle
assert xb.shape == (32, MAX_LEN) and yb.shape == (32,)
assert len(train_loader) == 114, f"114 lots attendus (3617 / 32, arrondi au-dessus), obtenu {len(train_loader)}"
assert torch.equal(next(iter(val_loader))[0], X_val[:32]), "val_loader ne doit pas mélanger"
print("✅ étape 5")

---
# Étape 6 — Le modèle

🎯 Écrire la classe `SpamClassifier`, un réseau en 3 blocs :

```
x : (lot, 40) numéros
   │  nn.Embedding(taille_vocab, 32, padding_idx=0)       chaque numéro → un vecteur de 32 nombres appris
   ▼
(lot, 40, 32)
   │  moyenne des vecteurs des VRAIS mots (on ignore les 0 du remplissage)
   ▼
(lot, 32)        un seul vecteur qui résume le SMS
   │  nn.Linear(32, 16) → nn.ReLU() → nn.Dropout(0.3) → nn.Linear(16, 1)
   ▼
(lot, 1) → .squeeze(1) → (lot,)      un score par SMS (un « logit »)
```

- `padding_idx=0` : le vecteur du numéro 0 reste nul et n'est jamais appris ;
- **pourquoi ignorer le remplissage dans la moyenne ?** Sinon un SMS de 5 mots aurait sa moyenne divisée par 40 au lieu de 5 : sa « signature » serait écrasée vers zéro, juste parce qu'il est court ;
- `Dropout(0.3)` : pendant l'entraînement, éteint au hasard 30 % des neurones, pour éviter que le réseau apprenne les SMS de train par cœur ;
- le réseau sort un **logit** (un score de −∞ à +∞), pas une probabilité : la fonction d'erreur de l'étape 7 applique elle-même la sigmoïde, ce qui est plus stable numériquement.

🧰 Outils : deux démos dans un autre contexte.

In [ ]:
# Démo 1 : la forme d'un nn.Module (autre contexte : 3 mesures → 1 score)
class MiniReseau(nn.Module):
    def __init__(self, n_entrees):
        super().__init__()                         # obligatoire, toujours en premier
        self.couche = nn.Linear(n_entrees, 1)      # les couches se déclarent dans __init__...

    def forward(self, x):                          # ...et s'enchaînent dans forward
        return self.couche(x).squeeze(1)

mini = MiniReseau(3)
print(mini(torch.randn(4, 3)).shape)               # 4 exemples → 4 scores

In [ ]:
# Démo 2 : moyenne en ignorant le remplissage
x = torch.tensor([[5, 8, 0, 0]])                                      # 1 SMS : 2 vrais mots + 2 zéros
vecteurs = torch.tensor([[[1., 1.], [3., 3.], [0., 0.], [0., 0.]]])   # leurs vecteurs (ici de taille 2)
masque = (x != 0).unsqueeze(-1).float()                               # 1 pour un vrai mot, 0 pour le remplissage
print("moyenne naïve      :", vecteurs.mean(dim=1))                                        # [1., 1.] : faux
print("moyenne sans les 0 :", (vecteurs * masque).sum(dim=1) / masque.sum(dim=1).clamp(min=1))  # [2., 2.] : juste

In [ ]:
# ✍️ À toi
# 1. class SpamClassifier(nn.Module) avec :
#    - __init__(self, taille_vocab, dim_emb=32) :
#        self.embedding = nn.Embedding(taille_vocab, dim_emb, padding_idx=0)
#        self.reseau = nn.Sequential(...)   les 4 couches du schéma
#    - forward(self, x) : embedding → moyenne sans le remplissage (démo 2) → reseau → squeeze(1)
# 2. modele = SpamClassifier(len(vocab))
# 3. n_parametres = le nombre total de poids du modèle ; affiche le modèle et n_parametres
#    (outil : sum(p.numel() for p in modele.parameters()))

In [ ]:
# ✅ Contrôle
sortie = modele(X_train[:5])
assert sortie.shape == (5,), f"forme attendue (5,), obtenue {tuple(sortie.shape)} : as-tu fait le squeeze(1) ?"
assert n_parametres == sum(p.numel() for p in modele.parameters())
assert n_parametres < 200_000, "trop de paramètres pour un « petit » réseau"
print(f"✅ étape 6 — {n_parametres:,} paramètres".replace(",", " "))

👀 **Observe** : quelle part des paramètres est dans l'embedding (`taille_vocab × 32`) ? Garde ce nombre de paramètres : tu le compareras à DistilBERT au template 3.

---
# Étape 7 — Entraîner

🎯 Deux choses à écrire :

**a. Une fonction `predire_probas(modele, loader)`** qui renvoie, pour tous les SMS d'un loader, la **probabilité de spam** (un tableau numpy de nombres entre 0 et 1). Elle servira à chaque évaluation.

**b. La boucle d'entraînement**, sur `N_EPOCHS = 30` époques :

```
pour chaque époque :
    modele.train()                               mode entraînement (Dropout actif)
    pour chaque lot (xb, yb) de train_loader :
        1. optimiseur.zero_grad()                efface les gradients du lot précédent
        2. logits = modele(xb)                   prédiction
        3. loss = critere(logits, yb)            erreur
        4. loss.backward()                       calcule dans quel sens corriger chaque poids
        5. optimiseur.step()                     corrige les poids
    puis, sans apprendre : loss sur val et F1 sur val (seuil 0,5)
    ajouter {"epoque", "loss_train", "loss_val", "f1_val"} à la liste historique
```

- `critere = nn.BCEWithLogitsLoss()` : l'erreur classique pour une classification **binaire** à partir de logits ;
- `optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3)` : l'algorithme qui corrige les poids, avec un pas (*learning rate*) de 0,001.

Mesure aussi la **durée** de l'entraînement dans `temps_entrainement` (en secondes) : elle servira à comparer le coût des modèles au template 4.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `modele.eval()` | mode évaluation (Dropout éteint) | à appeler avant de prédire |
| `with torch.no_grad():` | ne calcule pas de gradients (plus rapide, on n'apprend pas) | — |
| `torch.sigmoid(logits)` | logit → probabilité entre 0 et 1 | — |
| `torch.cat(liste_de_tenseurs)` | recolle les résultats des lots | — |
| `tenseur.numpy()` | tenseur → tableau numpy | — |
| `loss.item()` | la valeur de l'erreur, en nombre Python | — |
| `time.perf_counter()` | l'heure, en secondes (on fait fin − début) | — |
| `f1_score(vrais, predits)` | le F1 de la classe 1 (spam) | — |

In [ ]:
# ✍️ À toi
# a. def predire_probas(modele, loader) : eval, no_grad, sigmoid sur chaque lot, cat, numpy
#
# b. 1. N_EPOCHS = 30 ; critere = ... ; optimiseur = ... ; historique = []
#    2. debut = time.perf_counter()
#    3. la boucle sur les époques (schéma ci-dessus) ; loss_train = la moyenne des loss des lots
#       pour val : loss_val = critere(logits de tout X_val, y_val) ; f1_val avec un seuil de 0,5
#       affiche une ligne par époque
#    4. temps_entrainement = time.perf_counter() - debut

In [ ]:
# ✅ Contrôle
probas_test_controle = predire_probas(modele, val_loader)
assert probas_test_controle.shape == (776,) and probas_test_controle.min() >= 0 and probas_test_controle.max() <= 1, \
    "predire_probas doit renvoyer 776 probabilités entre 0 et 1"
assert len(historique) == N_EPOCHS and {"epoque", "loss_train", "loss_val", "f1_val"} <= set(historique[0])
assert historique[-1]["loss_train"] < historique[0]["loss_train"], "la loss de train ne baisse pas : le modèle n'apprend pas"
assert max(h["f1_val"] for h in historique) > 0.85, "F1 val trop bas : vérifie la moyenne sans le remplissage et l'optimiseur"
assert temps_entrainement > 0
print("✅ étape 7")

---
# Étape 8 — Lire les courbes d'apprentissage

🎯 Tracer `loss_train` et `loss_val` en fonction de l'époque, et diagnostiquer :

| Ce que tu vois | Diagnostic | Que faire |
|---|---|---|
| les deux baissent encore à la fin | **sous-apprentissage** : le modèle n'a pas fini d'apprendre | plus d'époques |
| train baisse, val **remonte** | **sur-apprentissage** : il apprend train par cœur | moins d'époques (arrêter au minimum de val) |
| les deux se stabilisent ensemble | c'est bon | — |

⚠️ La loss de val peut remonter alors que le F1 progresse encore : le modèle classe bien, mais devient **trop sûr de lui** sur ses erreurs. À toi de trancher, et de le justifier.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `pd.DataFrame(historique)` | la liste de dictionnaires devient un tableau | — |
| `px.line(df, x=..., y=[col1, col2])` | plusieurs courbes sur le même graphique | `markers=True` |

In [ ]:
# ✍️ À toi
# 1. courbes = pd.DataFrame(historique)
# 2. trace loss_train et loss_val en fonction de epoque, puis f1_val

✍️ **À écrire** : à quelle époque la loss de val est-elle la plus basse ? Y a-t-il sur-apprentissage ? Si oui, relance les étapes 6 et 7 (l'étape 6 recrée un modèle neuf) avec ce nombre d'époques.

*Ta réponse :*

---
# Étape 9 — Évaluer sur validation et choisir le seuil

🎯 Le modèle donne une **probabilité**. Pour décider « spam » ou « ham », il faut un **seuil** : par défaut 0,5, mais rien n'oblige à garder 0,5.

| Seuil | Effet |
|---|---|
| bas (0,2) | on bloque au moindre doute : **recall** ↑ (on rate peu de spams), **précision** ↓ (plus de vrais SMS bloqués) |
| haut (0,8) | on ne bloque que les cas sûrs : précision ↑, recall ↓ |

Le seuil se choisit **sur val, jamais sur test** : régler quoi que ce soit en regardant test, c'est tricher à l'examen.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `(probas >= seuil).astype(int)` | probabilités → 0 / 1 | — |
| `classification_report(vrais, predits)` | précision, recall, F1 de chaque classe | `target_names=["ham", "spam"]`, `digits=3` |
| `confusion_matrix(vrais, predits)` | le tableau vrais × prédits | lignes = vrai, colonnes = prédit |
| `precision_score`, `recall_score`, `f1_score` | une métrique pour la classe 1 (spam) | `zero_division=0` |
| `np.arange(0.1, 0.91, 0.05)` | les seuils de 0,10 à 0,90 par pas de 0,05 | — |

In [ ]:
# ✍️ À toi
# 1. probas_val = predire_probas(modele, val_loader)
# 2. au seuil 0,5 : affiche classification_report et confusion_matrix
# 3. tableau_seuils = un DataFrame avec, pour chaque seuil de 0,10 à 0,90 :
#    colonnes "seuil", "precision", "recall", "f1" (calculées sur val)
# 4. trace precision, recall et f1 en fonction du seuil
# 5. SEUIL = le seuil qui donne le meilleur F1 sur val (ou ton choix, argumenté ci-dessous)

In [ ]:
# ✅ Contrôle
assert {"seuil", "precision", "recall", "f1"} <= set(tableau_seuils.columns)
assert len(tableau_seuils) >= 10
assert 0 < SEUIL < 1
assert f1_score(val_df["label"], probas_val >= SEUIL) > 0.85
print(f"✅ étape 9 — F1 val au seuil {SEUIL} : {f1_score(val_df['label'], probas_val >= SEUIL):.3f}")

✍️ **À écrire** : quel seuil choisis-tu, et pourquoi ? Pense au métier : pour AT&T, qu'est-ce qui est le plus grave, un spam qui passe (faux négatif) ou le SMS d'un proche bloqué (faux positif) ?

*Ta réponse :*

---
# Étape 10 — Évaluation finale sur test (une seule fois)

🎯 Tout est réglé (époques, seuil) : on mesure **une seule fois** sur test, des SMS que le modèle n'a jamais vus et qui n'ont servi à aucun réglage. Ce sont **ces** chiffres qu'on annonce dans le livrable.

Ranger les résultats dans un dictionnaire `scores_embeddings` avec exactement ces clés :
`"modele"`, `"seuil"`, `"accuracy"`, `"precision"`, `"recall"`, `"f1"`, `"n_parametres"`, `"temps_entrainement_s"`.

Les métriques portent sur la classe **spam**. Convertis-les avec `float(...)` pour pouvoir les écrire en JSON à l'étape 11.

In [ ]:
# ✍️ À toi
# 1. probas_test = predire_probas(modele, test_loader) ; preds_test au SEUIL choisi
# 2. scores_embeddings = {"modele": "Embeddings + petit réseau", "seuil": SEUIL, ... les 8 clés}
# 3. affiche scores_embeddings et la matrice de confusion de test

In [ ]:
# ✅ Contrôle
cles = {"modele", "seuil", "accuracy", "precision", "recall", "f1", "n_parametres", "temps_entrainement_s"}
assert set(scores_embeddings) == cles, f"clés attendues : {cles}"
assert all(0 <= scores_embeddings[k] <= 1 for k in ["accuracy", "precision", "recall", "f1"])
assert probas_test.shape == (776,)
assert abs(scores_embeddings["f1"] - f1_score(test_df["label"], probas_test >= SEUIL)) < 1e-9, "le F1 doit être calculé au SEUIL choisi"
assert scores_embeddings["f1"] > 0.80
print("✅ étape 10")

---
# Étape 11 — Enregistrer pour le template 4

🎯 Écrire trois fichiers :

| Fichier | Contenu | Sur GitHub ? |
|---|---|---|
| `scores/scores_embeddings.json` | le dictionnaire `scores_embeddings` | ✅ oui : le template 4 en a besoin, même s'il est fait par un autre membre |
| `scores/predictions_embeddings.csv` | une ligne par SMS de test : `text`, `label`, `proba` | ✅ oui (petit fichier) |
| `models/embeddings.pt` | les poids du modèle | ❌ non (`.gitignore`) |

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `dossier.mkdir(exist_ok=True)` | crée le dossier s'il n'existe pas | `exist_ok=True` |
| `json.dump(objet, fichier, ...)` | écrit un dictionnaire en JSON | `indent=2`, `ensure_ascii=False` (garde les accents) |
| `open(chemin, "w", encoding="utf-8")` | ouvre un fichier en écriture | — |
| `df.assign(proba=...)` | ajoute une colonne | — |
| `torch.save(modele.state_dict(), chemin)` | enregistre les poids | — |

In [ ]:
# ✍️ À toi
# 1. crée DOSSIER_SCORES et DOSSIER_MODELES
# 2. écris scores_embeddings dans DOSSIER_SCORES / "scores_embeddings.json"
# 3. écris test_df + une colonne proba (= probas_test) dans DOSSIER_SCORES / "predictions_embeddings.csv" (sans index)
# 4. enregistre les poids dans DOSSIER_MODELES / "embeddings.pt"

In [ ]:
# ✅ Contrôle
with open(DOSSIER_SCORES / "scores_embeddings.json", encoding="utf-8") as f:
    relu = json.load(f)
assert relu == scores_embeddings
preds = pd.read_csv(DOSSIER_SCORES / "predictions_embeddings.csv")
assert list(preds.columns) == ["text", "label", "proba"] and len(preds) == 776
assert (preds["label"] == test_df["label"]).all(), "les lignes doivent rester dans l'ordre de test.csv"
assert (DOSSIER_MODELES / "embeddings.pt").exists()
print("✅ étape 11 : résultats prêts pour le template 4")

---
# Étape 12 — Essayer sur de nouveaux SMS

🎯 Écrire `predire_sms(texte)` qui renvoie la probabilité de spam d'**un** SMS inventé. Parfait pour une démo à l'oral.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `torch.tensor([encoder(texte)])` | un lot d'**un seul** SMS, forme `(1, 40)` | les crochets autour de `encoder(...)` |
| `tenseur.item()` | un tenseur d'un seul nombre → nombre Python | — |

In [ ]:
# ✍️ À toi
# 1. def predire_sms(texte) : eval, no_grad, encoder, modele, sigmoid → un nombre entre 0 et 1
# 2. pour chaque SMS de sms_inventes : affiche la probabilité et la décision au SEUIL ("SPAM" / "ham")
# 3. ajoute tes propres SMS à la liste : arrives-tu à tromper le modèle ?
sms_inventes = [
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT! You have won a 1000 cash prize. Call 09061234567 now to claim",
    "Ok see u later lol",
    "FREE entry: txt WIN to 80086 to receive your reward",
]

In [ ]:
# ✅ Contrôle
assert 0 <= predire_sms("hello") <= 1
assert predire_sms(sms_inventes[3]) > predire_sms(sms_inventes[0]), "le modèle devrait trouver le SMS « FREE entry » plus suspect que le déjeuner"
print("✅ étape 12 — template 2 terminé")

---
# 📝 Pour le livrable

Garde de côté, en une ligne chacun :

| Décision | Ta justification |
|---|---|
| tokenisation (minuscules, sans ponctuation, `numtoken`) | |
| vocabulaire construit sur train seulement | |
| `MIN_FREQ = 2` (taille du vocabulaire obtenue) | |
| `MAX_LEN = 40` avec remplissage | |
| embedding de 32 + moyenne sans le remplissage | |
| nombre d'époques retenu | |
| seuil retenu | |
| **scores sur test** (précision, recall, F1 spam) | |
| nombre de paramètres et durée d'entraînement | |